In [ ]:
import frontmatter

In [ ]:
with open('example.md', 'r', encoding='utf-8') as f:
    post = frontmatter.load(f)

In [ ]:
post.metadata['difficulty']

In [ ]:
post.content

In [ ]:
post.to_dict()

In [1]:
import io
import zipfile
import requests
import frontmatter

In [2]:
# https://github.com/evidentlyai/docs/archive/refs/heads/main.zip
url = 'https://github.com/evidentlyai/docs/archive/refs/heads/main.zip'
# url = 'https://github.com/DataTalksClub/faq/archive/refs/heads/main.zip'
resp = requests.get(url)

In [3]:
zf = zipfile.ZipFile(io.BytesIO(resp.content))

In [4]:
repository_data = []
for file_info in zf.infolist():
    filename = file_info.filename.lower()

    # Only process markdown files
    if not (filename.endswith('.md') or filename.endswith('.mdx')):
        continue

    # Read and parse each file
    with zf.open(file_info) as f_in:
        content = f_in.read()
        post = frontmatter.loads(content)
        data = post.to_dict()
        data['filename'] = filename
        repository_data.append(data)

zf.close()

In [8]:
len(repository_data[45]['content'])

21712

In [11]:
import io
import zipfile
import requests
import frontmatter
def read_repo_date(repo_owner, repo_name):
    url = f"https://github.com/{repo_owner}/{repo_name}/archive/refs/heads/main.zip"
    print(url)
    resp = requests.get(url)

    zf = zipfile.ZipFile(io.BytesIO(resp.content))

    repository_data = []
    for file_info in zf.infolist():
        filename = file_info.filename.lower()

        # Only process markdown files
        if not (filename.endswith('.md') or filename.endswith('.mdx')):
            continue

        # Read and parse each file
        with zf.open(file_info) as f_in:
            content = f_in.read()
            post = frontmatter.loads(content)
            data = post.to_dict()
            data['filename'] = filename
            repository_data.append(data)

    zf.close()
    return repository_data

In [13]:
# evidentlyai/docs
# DataTalksClub/faq
repo1 = ("evidentlyai", "docs")
repo_data = read_repo_date(*repo1)
# repo_data

https://github.com/evidentlyai/docs/archive/refs/heads/main.zip


In [14]:
def sliding_window(seq, size, step):
    if size <= 0 or step <= 0:
        raise ValueError("size and step must be positive")

    n = len(seq)
    result = []
    for i in range(0, n, step):
        chunk = seq[i:i+size]
        result.append({'start': i, 'chunk': chunk})
        if i + size >= n:
            break

    return result

In [15]:
evidently_chunks = []

for doc in repo_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    chunks = sliding_window(doc_content, 2000, 1000)
    for chunk in chunks:
        chunk.update(doc_copy)
    evidently_chunks.extend(chunks)

In [16]:
evidently_chunks[:5]

[{'start': 0,
  'chunk': '<Note>\n  If you\'re not looking to build API reference documentation, you can delete\n  this section by removing the api-reference folder.\n</Note>\n\n## Welcome\n\nThere are two ways to build API documentation: [OpenAPI](https://mintlify.com/docs/api-playground/openapi/setup) and [MDX components](https://mintlify.com/docs/api-playground/mdx/configuration). For the starter kit, we are using the following OpenAPI specification.\n\n<Card\n  title="Plant Store Endpoints"\n  icon="leaf"\n  href="https://github.com/mintlify/starter/blob/main/api-reference/openapi.json"\n>\n  View the OpenAPI specification file\n</Card>\n\n## Authentication\n\nAll API endpoints are authenticated using Bearer tokens and picked up from the specification file.\n\n```json\n"security": [\n  {\n    "bearerAuth": []\n  }\n]\n```',
  'title': 'Introduction',
  'description': 'Example section for showcasing API endpoints',
  'filename': 'docs-main/api-reference/introduction.mdx'},
 {'start'

In [17]:
len(evidently_chunks)

573

## Paragraph splitting

In [18]:
sample_content = repo_data[45]['content']
sample_content[:10]

'In this tu'

In [26]:
import re
def split_markdown_by_level(text, level=2):
    """
    Split markdown text by a specific header level.

    :param text: Markdown text as a string
    :param level: Header level to split on
    :return: List of sections as strings
    """
    # This regex matches markdown headers
    # For level 2, it matches lines starting with "## "
    header_pattern = r'^(#{' + str(level) + r'} )(.+)$'
    pattern = re.compile(header_pattern, re.MULTILINE)

    # Split and keep the headers
    parts = pattern.split(text)

    sections = []
    for i in range(1, len(parts), 3):
        # We step by 3 because regex.split() with
        # capturing groups returns:
        # [before_match, group1, group2, after_match, ...]
        # here group1 is "## ", group2 is the header text
        header = parts[i] + parts[i+1]  # "## " + "Title"
        header = header.strip()

        # Get the content after this header
        content = ""
        if i+2 < len(parts):
            content = parts[i+2].strip()

        if content:
            section = f'{header}\n\n{content}'
        else:
            section = header
        sections.append(section)

    return sections

In [28]:
sample_section = split_markdown_by_level(sample_content)
sample_section[0]

'## 1. Installation and Imports\n\nInstall Evidently:\n\n```python\npip install evidently[llm] \n```\n\nImport the required modules:\n\n```python\nimport pandas as pd\nfrom evidently.future.datasets import Dataset\nfrom evidently.future.datasets import DataDefinition\nfrom evidently.future.datasets import Descriptor\nfrom evidently.future.descriptors import *\nfrom evidently.future.report import Report\nfrom evidently.future.presets import TextEvals\nfrom evidently.future.metrics import *\nfrom evidently.future.tests import *\n\nfrom evidently.features.llm_judge import BinaryClassificationPromptTemplate\n```\n\nTo connect to Evidently Cloud:\n\n```python\nfrom evidently.ui.workspace.cloud import CloudWorkspace\n```\n\n**Optional.** To create monitoring panels as code:\n\n```python\nfrom evidently.ui.dashboards import DashboardPanelPlot\nfrom evidently.ui.dashboards import DashboardPanelTestSuite\nfrom evidently.ui.dashboards import DashboardPanelTestSuiteCounter\nfrom evidently.ui.dash

In [29]:
evidently_chunks = []

for doc in repo_data:
    doc_copy = doc.copy()
    doc_content = doc_copy.pop('content')
    sections = split_markdown_by_level(doc_content, level=2)
    for section in sections:
        section_doc = doc_copy.copy()
        section_doc['section'] = section
        evidently_chunks.append(section_doc)

In [30]:
evidently_chunks[2]

{'title': 'Data definition',
 'description': 'How to map the input data.',
 'filename': 'docs-main/docs/library/data_definition.mdx',
 'section': '## Basic flow\n\n**Step 1. Imports.** Import the following modules:\n\n```python\nfrom evidently import Dataset\nfrom evidently import DataDefinition\n```\n\n**Step 2. Prepare your data.** Use a pandas.DataFrame.\n\n<Info>\n  Your data can have [flexible structure](/docs/library/overview#dataset) with any mix of categorical, numerical or text columns. Check the [Reference table](/metrics/all_metrics) for data requirements in specific evaluations.\n</Info>\n\n**Step 3. Create a Dataset object**. Use `Dataset.from_pandas` with `data_definition`:\n\n```python\neval_data = Dataset.from_pandas(\n    source_df,\n    data_definition=DataDefinition()\n)\n```\n\nTo map columns automatically, pass an empty `DataDefinition()` . Evidently will map columns:\n\n- By type (numerical, categorical).\n- By matching column names to roles (e.g., a column "targe